# Exploring SnapBoost Parameters

SnapBoost stochastically selects base learners each boosting round:

- **Decision trees** at various depths (local, axis-aligned structure)
- **Kernel ridge** with an RBF kernel (smooth, global patterns)

This notebook compares tree-only vs. mixed ensembles and sweeps a few key hyperparameters on a synthetic dataset with both piecewise and smooth structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split

from snapboost import SnapBoost

## Synthetic dataset

The target combines a step function (tree-friendly) with a smooth sinusoidal component (ridge-friendly).

In [ ]:
rng = np.random.default_rng(42)
n_samples = 800

X = rng.uniform(-3, 3, size=(n_samples, 2))
y = (
    np.where(X[:, 0] > 0, 2.0, -1.0)
    + 0.5 * np.sin(2 * X[:, 1])
    + rng.normal(0, 0.15, size=n_samples)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

## Compare `p_tree`: trees only vs. mixed ensemble

Setting `p_tree=1.0` uses only decision trees; `p_tree=0.5` gives equal weight to trees and the kernel ridge learner.

In [ ]:
def train_and_evaluate(p_tree, **kwargs):
    model = SnapBoost(
        num_iterations=80,
        learning_rate=0.1,
        p_tree=p_tree,
        min_max_depth=3,
        max_max_depth=6,
        mode="regression",
        random_state=42,
        verbose=False,
        **kwargs,
    )
    model.fit(X_train, y_train)
    r2 = model.score(X_test, y_test)
    rmse = model.evaluate(X_test, y_test)
    return {"p_tree": p_tree, "r2": r2, "rmse": rmse}


results = []
for p_tree in [1.0, 0.8, 0.5, 0.2, 0.0]:
    print(f"\n--- p_tree={p_tree} ---")
    results.append(train_and_evaluate(p_tree))

pd.DataFrame(results).set_index("p_tree")

## Effect of tree depth range

Shallow trees capture broad splits; deeper trees fit finer local structure.

In [ ]:
depth_configs = [
    (2, 3, "Shallow (2-3)"),
    (4, 8, "Default (4-8)"),
    (6, 12, "Deep (6-12)"),
]

depth_results = []
for min_d, max_d, label in depth_configs:
    model = SnapBoost(
        num_iterations=80,
        learning_rate=0.1,
        p_tree=0.8,
        min_max_depth=min_d,
        max_max_depth=max_d,
        mode="regression",
        random_state=42,
        verbose=False,
    )
    model.fit(X_train, y_train)
    depth_results.append({
        "config": label,
        "min_max_depth": min_d,
        "max_max_depth": max_d,
        "r2": model.score(X_test, y_test),
        "rmse": model.evaluate(X_test, y_test),
    })

pd.DataFrame(depth_results).set_index("config")

## Kernel ridge parameters (`alpha`, `gamma`)

These only affect the ridge learner selected with probability `1 - p_tree`.

In [ ]:
kernel_results = []
for alpha, gamma in [(0.1, 0.5), (1.0, 1.0), (10.0, 2.0)]:
    model = SnapBoost(
        num_iterations=80,
        learning_rate=0.1,
        p_tree=0.5,
        alpha=alpha,
        gamma=gamma,
        mode="regression",
        random_state=42,
        verbose=False,
    )
    model.fit(X_train, y_train)
    kernel_results.append({
        "alpha": alpha,
        "gamma": gamma,
        "r2": model.score(X_test, y_test),
        "rmse": model.evaluate(X_test, y_test),
    })

pd.DataFrame(kernel_results)

## Visualize predictions along one axis

Fix `x1 = 0` and plot the learned function along `x2`.

In [ ]:
x2_grid = np.linspace(-3, 3, 200)
X_line = np.column_stack([np.zeros_like(x2_grid), x2_grid])
y_line_true = np.where(0 > 0, 2.0, -1.0) + 0.5 * np.sin(2 * x2_grid)

configs = [
    ("Trees only (p_tree=1.0)", {"p_tree": 1.0}),
    ("Mixed (p_tree=0.5)", {"p_tree": 0.5}),
    ("Ridge only (p_tree=0.0)", {"p_tree": 0.0}),
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x2_grid, y_line_true, "k--", linewidth=2, label="True (noise-free)")

for label, params in configs:
    model = SnapBoost(
        num_iterations=80,
        learning_rate=0.1,
        min_max_depth=3,
        max_max_depth=6,
        mode="regression",
        random_state=42,
        verbose=False,
        **params,
    )
    model.fit(X_train, y_train)
    ax.plot(x2_grid, model.predict(X_line), linewidth=1.5, label=label)

ax.set_xlabel("x2")
ax.set_ylabel("Predicted y")
ax.set_title("SnapBoost predictions at x1 = 0")
ax.legend()
plt.tight_layout()
plt.show()

## Custom HNBM with a single learner type

For full control over the learner pool, subclass `HNBM` and set `base_learners_` and `probabilities_` before calling `fit`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from snapboost import HNBM


class TreeOnlyBoost(HNBM):
    def __init__(self, max_depth=5, **kwargs):
        super().__init__(**kwargs)
        self.base_learners_ = [DecisionTreeRegressor(max_depth=max_depth, random_state=42)]
        self.probabilities_ = [1.0]


custom = TreeOnlyBoost(
    max_depth=5,
    num_iterations=80,
    learning_rate=0.1,
    mode="regression",
    random_state=42,
    verbose=False,
)
custom.fit(X_train, y_train)

print(f"R²:   {custom.score(X_test, y_test):.4f}")
custom.evaluate(X_test, y_test)